# Hybrid ML + DL model

**Owner:** Kapish Rohilla. Phase 2 (deep learning), step 4 of 5.

## Purpose

Phase 1 tree models and Phase 2 sequence models see the engine history in different ways. The Random Forest uses 57 hand-made features over the last 10 cycles; the LSTM, GRU, 1D-CNN and Transformer read a raw 30-cycle window. This notebook tests whether a **hybrid** that combines the two is better than either part, and whether any gain is worth the added complexity compared with a plain average.

Phase 1 weaknesses this targets:
1. every Phase 1 model underpredicts engines with true RUL above 200 cycles;
2. the 10-cycle rolling features only see recent behaviour;
3. RMSE is much larger than MAE, so a few large errors dominate.

## Method

### Design (written before any experiment)

**Hybrid A (stacking).** A deep model $f_\theta$ maps the 30-cycle window $x_{t-29:t}$ of an engine to a RUL estimate

$$d_t = f_\theta(x_{t-29:t}).$$

Phase 1's rolling features $\phi(x)_t \in \mathbb{R}^{57}$ (cycle, 14 sensors, and the 10-cycle mean, standard deviation and slope of each sensor) are joined with $d_t$, and a tree regressor $g$ (Random Forest or Gradient Boosting, Phase 1 hyperparameters, not retuned) predicts

$$\hat y_t = g\big([\phi(x)_t,\ d_t]\big).$$

**Hybrid B (average, no fitted weights).** $\hat y_t = \tfrac12\big(d^{\text{LSTM}}_t + \hat y^{\text{RF}}_t\big)$, the mean of the best DL model (`lstm_v1`) and the best Phase 1 model (Random Forest). If B is as good as A, the stacking is not justified.

**Hybrid C (residual).** The tree predicts the DL error $r_t = y_t - d_t$ from the same features, and $\hat y_t = d_t + g([\phi(x)_t, d_t])$.

**Why a scalar $d$ and not the embedding.** Each trained network has its own embedding coordinates, so the 5 fold networks below produce embeddings that cannot be mixed in one tree column. A RUL estimate in cycles means the same thing for every network. A bottleneck-embedding variant was left out for this reason and for time.

### Hypothesis

The sequence model summarises a longer history (30 cycles) than the rolling features (10 cycles), and the tree mixes it nonlinearly with the engineered features and can correct the DL model where it is systematically wrong. So Hybrid A should beat both of its parts on the validation engines, especially for high true RUL, and should beat the plain average B.

### Leakage control: out-of-fold predictions by engine

If the tree were trained on $d$ from an LSTM that was itself trained on the same 80 engines, $d$ would be far more accurate on those rows than on new engines, the tree would learn to trust it too much, and validation error would rise. So the training values of $d$ are **out-of-fold**:

1. The 80 training engines are split into $K=5$ folds **by engine** (`hybrid.engine_folds`, seeded NumPy permutation).
2. For fold $k$, a fresh LSTM with the `config_lstm.json` settings is trained on the other 64 engines: 56 for gradient steps and 8 for early stopping (`hybrid.split_fit_stop`). The scaler is fitted on the 56 engines. Fold $k$ and the 20 validation engines are never seen.
3. The model predicts the 16 engines of fold $k$. Every training row therefore gets a $d$ from a model that never saw its engine.
4. **Validation and test $d$** = mean of the 5 fold models. We chose this over the saved `lstm_v1` because (a) the fold models never saw the validation engines, while `lstm_v1` used them for early stopping, so selecting on them would be optimistic; (b) each fold model was trained on 64 engines, like the models behind the out-of-fold training values, so training and inference $d$ come from the same kind of model; (c) the whole procedure can be repeated per seed. The same choice is used for validation and test. Remaining mismatch: a mean of 5 models is smoother and slightly more accurate than one out-of-fold model.

`tests/test_hybrid.py` checks fold assignment by engine, that no fold model saw its held-out engines or any validation engine, that test data (no `rul`) is refused, and that all joins are by `(engine_id, cycle)`.

### Variants and selection rule (fixed before running)

| Variant | Features | Tree | Target |
| --- | --- | --- | --- |
| A-RF | rolling + $d_{\text{LSTM}}$ | Random Forest | RUL |
| A-GB | rolling + $d_{\text{LSTM}}$ | Gradient Boosting | RUL |
| A-RF-2DL | rolling + $d_{\text{LSTM}}$ + $d_{\text{GRU}}$ | Random Forest | RUL |
| C-RF | rolling + $d_{\text{LSTM}}$ | Random Forest | $y - d_{\text{LSTM}}$ |
| B | – | – | mean of `lstm_v1` and Phase 1 RF |
| B-fold | – | – | mean of fold-mean $d_{\text{LSTM}}$ and Phase 1 RF |

- The DL model is the one with the lowest `best_val_window_mae` in the four `config_*.json` files (the Yogesh rule); the second-best recurrent model supplies $d_{\text{GRU}}$.
- **Selection:** lowest MAE on all windows of the 20 validation engines; ties within 0.05 cycles go to the simpler variant. The 100 fixed points are reported once, after selection, and are not used to choose.
- Training target is the true, uncapped RUL; evaluation always uses the true RUL.
- B-fold uses the same $d$ as A and C, so B versus B-fold separates the effect of averaging from the effect of using `lstm_v1` (which saw the validation engines during early stopping).
- Seeds 42, 43, 44 repeat the whole procedure (folds, fold LSTMs and GRUs, trees) for every stacked variant and B-fold; B changes only through the Random Forest seed. Saved predictions are seed 42.
- Output columns: `hybrid_v1` = best Hybrid A variant, `hybrid_resid_v1` = C-RF, `hybrid_avg_v1` = B.
- **Runtime limits:** fold models use their config budget (at most 60 epochs, patience 8) on CPU; 30 fold networks in total (5 LSTMs and 5 GRUs per seed). No test metric is computed here.

### Setup

Imports, paths, and the random seed. Torch runs on CPU, which is deterministic and as fast as MPS for these small models (see `02_yogesh` handoff).

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve().parents[1]  # run from notebooks/phase2
sys.path.insert(0, str(ROOT / "src"))

import hybrid
import train
from evaluation import VAL_ENGINES, split_train_val, test_points, validation_points
from hybrid import KEYS
from load_data import load_test, load_train

RESULTS = ROOT / "results" / "phase2"
FIGURES = RESULTS / "figures"
SEED = 42
SEEDS = [42, 43, 44]
START = time.perf_counter()
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 11})

### Data

The 80/20 engine split from Phase 1, the 100 fixed validation points and the 100 test points (last observed cycle of each test engine). Rows are kept as raw data; rolling features and windows are built per engine inside `src/hybrid.py`.

In [ ]:
train_df, val_df = split_train_val(load_train())
test_df = load_test()
val_pts = validation_points(val_df)  # 100 rows, fixed order
test_pts = test_points(test_df)[KEYS].reset_index(drop=True)  # engine 1..100

y_train = train_df.sort_values(KEYS)["rul"].to_numpy(dtype=float)  # same order as the hybrid tables
y_val_win = val_df.sort_values(KEYS)["rul"].to_numpy(dtype=float)
y_val_pts = val_pts["rul"].to_numpy(dtype=float)

assert set(train_df.engine_id).isdisjoint(VAL_ENGINES) and len(val_pts) == len(test_pts) == 100
print(f"training engines {train_df.engine_id.nunique()}, rows {len(train_df):,}")
print(f"validation engines {val_df.engine_id.nunique()}, rows {len(val_df):,}; test engines {test_df.engine_id.nunique()}")

### Check: saved DL models reproduce their reported scores

Before any DL output is used, each saved model from steps 2 and 3 is reloaded and scored on the 100 points. Its MAE must match its `config_*.json` value. The model with the lowest validation-window MAE (selection rule) becomes the hybrid's DL component.

In [ ]:
def load_config(kind: str) -> dict:
    """Read config_<kind>.json and return it with constructor arguments under 'kwargs'."""
    cfg = json.loads((RESULTS / f"config_{kind}.json").read_text())
    if "kwargs" not in cfg:  # the recurrent configs store them at the top level
        cfg["kwargs"] = {k: cfg[k] for k in ("hidden", "layers", "dropout")}
    return cfg


configs = {kind: load_config(kind) for kind in ["lstm", "gru", "cnn1d", "transformer"]}
check_rows = []
for kind, cfg in configs.items():
    model = train.load_model(kind, RESULTS / "models" / f"{kind}_v1.pt", **cfg["kwargs"])
    pred = hybrid.dl_predict(model, hybrid.sequences.fit_scaler(train_df), val_df, cfg["window"])
    pred = val_pts[KEYS].merge(pred, on=KEYS, how="left")
    check_rows.append({
        "model": cfg["model"],
        "selection_window_mae": cfg["best_val_window_mae"],
        "reported_mae": cfg["val_points"]["mae"],
        "reloaded_mae": hybrid.point_metrics(y_val_pts, pred["d"])["mae"],
    })
reload_check = pd.DataFrame(check_rows).sort_values("selection_window_mae")
assert np.allclose(reload_check.reported_mae, reload_check.reloaded_mae, atol=0.01)
DL_KIND, DL2_KIND = "lstm", "gru"
assert reload_check.model.iloc[0] == f"{DL_KIND}_v1"  # best by the selection rule
reload_check.round(3)

### Check: Phase 1 Random Forest is reproduced

The rolling-features-only tree from `hybrid.make_tree` must give the same predictions as Phase 1 (`results/phase1/val_predictions_trees.csv`). This confirms the tree settings were copied correctly. Its predictions are also the tree half of Hybrid B.

In [ ]:
X_roll_train = hybrid.build_hybrid_features(train_df, None, groups=["rolling"])
rf_phase1 = hybrid.fit_hybrid(X_roll_train, y_train, tree="random_forest", seed=SEED)
rf_val_pts = hybrid.predict_hybrid(
    rf_phase1, hybrid.build_hybrid_features(val_df, None, ["rolling"], points=val_pts)
)
phase1_trees = pd.read_csv(ROOT / "results" / "phase1" / "val_predictions_trees.csv")
assert phase1_trees[KEYS].equals(val_pts[KEYS])
print("max |difference| to Phase 1 RF:", np.abs(rf_val_pts - phase1_trees.random_forest).max())
print("Phase 1 RF MAE:", round(hybrid.point_metrics(y_val_pts, rf_val_pts)["mae"], 2))

## Results

### Out-of-fold LSTM predictions (seed 42)

`dl_stage` runs the out-of-fold procedure for one DL model and seed. It returns the out-of-fold $d$ for the 80 training engines, and the fold-mean $d$ for every validation and test row. The fold LSTMs use the `config_lstm.json` architecture and training settings unchanged.

In [ ]:
def fit_kwargs(cfg: dict) -> dict:
    """Training settings of a saved DL config, as keyword arguments for train.fit."""
    out = {k: cfg[k] for k in ("lr", "weight_decay", "batch_size", "grad_clip", "patience", "restore_best")}
    return {**out, "epochs": cfg["max_epochs"]}


def dl_stage(kind: str, seed: int, column: str = "d") -> dict:
    """Out-of-fold d on training rows and fold-mean d on validation and test rows."""
    cfg = configs[kind]
    oof, folds = hybrid.oof_predictions(
        train_df, kind, cfg["kwargs"], window=cfg["window"], k=5, seed=seed, n_stop=8,
        fit_kwargs=fit_kwargs(cfg),
    )
    rename = {"d": column}
    return {
        "train": oof.drop(columns="fold").rename(columns=rename),
        "val": hybrid.fold_mean_predict(folds, val_df, cfg["window"]).rename(columns=rename),
        "test": hybrid.fold_mean_predict(folds, test_df, cfg["window"]).rename(columns=rename),
        "oof": oof,
        "folds": folds,
    }


t0 = time.perf_counter()
lstm42 = dl_stage(DL_KIND, SEED)
print(f"5 fold LSTMs: {time.perf_counter() - t0:.0f} s")
pd.DataFrame([
    {
        "fold": i,
        "fit_engines": len(fm.fit_engines),
        "stop_engines": len(fm.stop_engines),
        "holdout_engines": len(fm.holdout_engines),
        "epochs_run": fm.history["epochs_run"][0],
        "best_epoch": fm.history["best_epoch"][0],
    }
    for i, fm in enumerate(lstm42["folds"])
])

The table shows each fold model trained on 56 engines, early-stopped on 8 others, and predicting 16 unseen engines. Below, the out-of-fold $d$ is compared with the error of the saved `lstm_v1` on its own training rows (in-sample). The gap is the optimism a tree would learn from if $d$ were not out-of-fold.

In [ ]:
lstm_v1 = train.load_model(DL_KIND, RESULTS / "models" / f"{DL_KIND}_v1.pt", **configs[DL_KIND]["kwargs"])
full_scaler = hybrid.sequences.fit_scaler(train_df)  # the scaler lstm_v1 was trained with
lstm_v1_pred = {
    "train": hybrid.dl_predict(lstm_v1, full_scaler, train_df),
    "val": hybrid.dl_predict(lstm_v1, full_scaler, val_df),
    "test": hybrid.dl_predict(lstm_v1, full_scaler, test_df),
}
assert lstm_v1_pred["train"][KEYS].equals(lstm42["train"][KEYS])

d_quality = pd.DataFrame([
    {"d on 80 training engines": "out-of-fold (fold models)",
     "mae": hybrid.point_metrics(y_train, lstm42["train"]["d"])["mae"]},
    {"d on 80 training engines": "in-sample (lstm_v1)",
     "mae": hybrid.point_metrics(y_train, lstm_v1_pred["train"]["d"])["mae"]},
    {"d on 80 training engines": "validation windows, fold mean (reference)",
     "mae": hybrid.point_metrics(y_val_win, lstm42["val"]["d"])["mae"]},
])
d_quality.round(2)

### Second DL model (GRU, seed 42)

The GRU is the runner-up recurrent model by the selection rule. Its out-of-fold estimate becomes a second column `d_gru` for variant A-RF-2DL. It tests whether two different sequence summaries help the tree more than one.

In [ ]:
t0 = time.perf_counter()
gru42 = dl_stage(DL2_KIND, SEED, column="d_gru")
print(f"5 fold GRUs: {time.perf_counter() - t0:.0f} s")
print("GRU out-of-fold MAE on training rows:", round(hybrid.point_metrics(y_train, gru42["train"]["d_gru"])["mae"], 2))

### Fit and score every variant

`run_variants` builds the hybrid tables for one seed and returns each variant's predictions on three sets:
- all validation-engine windows, used for selection;
- the 100 fixed points, used for reporting;
- the 100 test points, written to file only.

Every table is joined on `(engine_id, cycle)` inside `hybrid.build_hybrid_features`. The Random Forest on rolling features alone (the Phase 1 model) is refitted with the same seed for Hybrids B and B-fold.

In [ ]:
STACKED = {  # name: (DL columns, tree, residual)
    "A-RF": (["d"], "random_forest", False),
    "A-GB": (["d"], "gradient_boosting", False),
    "A-RF-2DL": (["d", "d_gru"], "random_forest", False),
    "C-RF": (["d"], "random_forest", True),
}
SIMPLICITY = ["B", "B-fold", "A-RF", "A-GB", "C-RF", "A-RF-2DL"]  # tie-break order, simplest first
SETS = {"val_win": (val_df, None), "val_pts": (val_df, val_pts), "test": (test_df, test_pts)}
SPLIT_OF = {"val_win": "val", "val_pts": "val", "test": "test"}


def merge_dl(*stages: dict) -> dict:
    """Join the d columns of several dl_stage results per split, by (engine_id, cycle)."""
    out = {}
    for split in ("train", "val", "test"):
        frame = stages[0][split]
        for other in stages[1:]:
            frame = frame.merge(other[split], on=KEYS, how="inner", validate="one_to_one")
        assert len(frame) == len(stages[0][split])
        out[split] = frame
    return out


def predict_sets(model: hybrid.HybridModel, dl: dict, groups: list) -> dict:
    """Predictions of a fitted tree on validation windows, validation points and test points."""
    out = {}
    for name, (df, points) in SETS.items():
        frame = dl[SPLIT_OF[name]] if dl is not None else None
        out[name] = hybrid.predict_hybrid(model, hybrid.build_hybrid_features(df, frame, groups, points))
    return out


def dl_sets(dl_split: dict, column: str = "d") -> dict:
    """A DL prediction itself on validation windows, validation points and test points."""
    out = {}
    for name, (_, points) in SETS.items():
        frame = dl_split[SPLIT_OF[name]]
        if points is not None:
            frame = points[KEYS].merge(frame, on=KEYS, how="left", validate="one_to_one")
        out[name] = frame[column].to_numpy()
    return out


def run_variants(dl: dict, seed: int, names: list) -> dict:
    """Predictions of each named variant (and of its parts) for one seed."""
    rf = hybrid.fit_hybrid(X_roll_train, y_train, "random_forest", seed=seed)
    preds = {"RF (Phase 1)": predict_sets(rf, None, ["rolling"]),
             "LSTM fold mean": dl_sets(dl),
             "lstm_v1": dl_sets(lstm_v1_pred)}
    for name in names:
        if name in STACKED:
            cols, tree, residual = STACKED[name]
            frame = {split: dl[split][KEYS + cols] for split in ("train", "val", "test")}
            X = hybrid.build_hybrid_features(train_df, frame["train"])
            model = hybrid.fit_hybrid(X, y_train, tree, residual=residual, seed=seed)
            preds[name] = predict_sets(model, frame, list(hybrid.FEATURE_GROUPS))
            preds[name]["model"] = model
    for name, dl_name in [("B", "lstm_v1"), ("B-fold", "LSTM fold mean")]:
        preds[name] = {s: hybrid.average_hybrid(preds[dl_name][s], preds["RF (Phase 1)"][s]) for s in SETS}
    return preds


def score_table(preds: dict) -> pd.DataFrame:
    """Selection score (validation windows) and 100-point metrics for every entry of preds."""
    rows = []
    for name, p in preds.items():
        rows.append({"model": name,
                     "window_mae": hybrid.point_metrics(y_val_win, p["val_win"])["mae"],
                     **hybrid.point_metrics(y_val_pts, p["val_pts"])})
    return pd.DataFrame(rows)


t0 = time.perf_counter()
dl42 = merge_dl(lstm42, gru42)
preds42 = run_variants(dl42, SEED, list(STACKED))
print(f"variants fitted: {time.perf_counter() - t0:.0f} s")

### Selection on the 20 validation engines (seed 42)

The rule: lowest MAE over all windows of the 20 validation engines, with ties within 0.05 cycles going to the simpler variant. The parts are listed for reference but are not candidates. `lstm_v1` (and therefore B) used these engines for early stopping, so their window scores are slightly optimistic. The 100-point columns are shown only after the choice is made.

In [ ]:
scores42 = score_table(preds42)
candidates = scores42[scores42.model.isin(SIMPLICITY)].copy()
best_score = candidates.window_mae.min()
tied = candidates[candidates.window_mae <= best_score + 0.05].model
CHOSEN = min(tied, key=SIMPLICITY.index)
BEST_A = min((n for n in STACKED if n.startswith("A-")), key=lambda n: scores42.set_index("model").window_mae[n])
print(f"chosen hybrid: {CHOSEN} (tied within 0.05: {list(tied)}); best Hybrid A: {BEST_A}")
scores42.sort_values("window_mae").round(2)

### Seeds 42, 43, 44

The whole procedure (fold assignment, 5 fold LSTMs, 5 fold GRUs, trees) is repeated for each seed, for every variant. The table gives mean and standard deviation (`ddof=1`) over the three seeds. These spreads are the yardstick for every difference discussed below.

In [ ]:
seed_scores = [scores42.assign(seed=SEED)]
t0 = time.perf_counter()
for seed in SEEDS[1:]:
    dl = merge_dl(dl_stage(DL_KIND, seed), dl_stage(DL2_KIND, seed, column="d_gru"))
    seed_scores.append(score_table(run_variants(dl, seed, list(STACKED))).assign(seed=seed))
print(f"seeds 43 and 44: {time.perf_counter() - t0:.0f} s")
seed_scores = pd.concat(seed_scores, ignore_index=True)
seed_summary = seed_scores.groupby("model", sort=False).agg(
    window_mae=("window_mae", "mean"), window_mae_std=("window_mae", "std"),
    mae=("mae", "mean"), mae_std=("mae", "std"),
    rmse=("rmse", "mean"), rmse_std=("rmse", "std"),
).sort_values("window_mae")
seed_summary.round(2)

### Hybrid versus its parts and versus the simple average

Each difference below is measured against the seed standard deviations above. The 100-point MAE is shown alongside, but selection used only the validation-engine windows.

In [ ]:
def seed_stats(name: str) -> pd.Series:
    """Mean and std over seeds of the window MAE and 100-point MAE of one model."""
    return seed_summary.loc[name, ["window_mae", "window_mae_std", "mae", "mae_std"]]


comparison = pd.DataFrame({name: seed_stats(name) for name in
                           [CHOSEN, "B", "B-fold", "C-RF", "lstm_v1", "LSTM fold mean", "RF (Phase 1)"]}).T
for ref in ["RF (Phase 1)", "lstm_v1", "B"]:
    comparison[f"window_gain_vs_{ref}"] = comparison.loc[ref, "window_mae"] - comparison["window_mae"]
comparison.round(2)

### Error by true-RUL band (seed 42)

Mean signed error (prediction minus truth, in cycles) per band of true RUL, with the number of validation points in each band. A negative value means the model underestimates the remaining life, as every Phase 1 model did above 200 cycles. The 200+ band has only 5 points, so differences there are indicative only.

In [ ]:
band_models = {
    "RF (Phase 1)": preds42["RF (Phase 1)"]["val_pts"],
    "lstm_v1": preds42["lstm_v1"]["val_pts"],
    f"hybrid ({CHOSEN})": preds42[CHOSEN]["val_pts"],
    "B (average)": preds42["B"]["val_pts"],
}
bands = hybrid.band_errors(y_val_pts, band_models)
bands.round(1)

### Component ablation (Random Forest, seed 42)

The same Random Forest is trained on three feature sets: the out-of-fold LSTM estimate alone, the Phase 1 rolling features alone (this is the Phase 1 model), and both together (variant A-RF). This shows which part carries the signal. Vikash's full ablations reuse these functions (`groups=` in `hybrid.build_hybrid_features`).

In [ ]:
component_rows = []
for label, groups in [("dl only", ["dl"]), ("rolling only", ["rolling"]), ("both", ["rolling", "dl"])]:
    frame = {split: dl42[split][KEYS + ["d"]] for split in ("train", "val", "test")}
    X = hybrid.build_hybrid_features(train_df, frame["train"], groups)
    model = hybrid.fit_hybrid(X, y_train, "random_forest", seed=SEED)
    p = predict_sets(model, frame, groups)
    component_rows.append({"components": label, "features": X.shape[1],
                           "window_mae": hybrid.point_metrics(y_val_win, p["val_win"])["mae"],
                           **hybrid.point_metrics(y_val_pts, p["val_pts"])})
components = pd.DataFrame(component_rows)
components.round(2)

### What the tree relies on

The Random Forest's impurity importances for the chosen hybrid (seed 42), top 10. If the DL columns rank at the top, the tree is mostly refining the sequence models' estimates rather than replacing them.

In [ ]:
chosen_model = preds42[CHOSEN]["model"]
importance = pd.Series(chosen_model.tree.feature_importances_, index=chosen_model.columns)
importance.sort_values(ascending=False).head(10).round(3)

### Negative control: a deliberately leaky hybrid (not a result)

This shows what out-of-fold stacking prevents. Two trees with the A-RF setup are trained. Both are then evaluated with the **same** validation inputs ($d$ from `lstm_v1`):
- **leaky:** trained on in-sample $d$, meaning `lstm_v1`'s predictions on the 80 engines it was trained on;
- **out-of-fold:** trained on the out-of-fold $d$, as in the real hybrid.

The only difference is how the training $d$ was produced. The leaky tree is a diagnostic and is never reported as a model.

In [ ]:
def control_run(train_d: pd.DataFrame) -> dict:
    """A-RF trained with the given training d, evaluated with lstm_v1's d on validation."""
    X = hybrid.build_hybrid_features(train_df, train_d)
    model = hybrid.fit_hybrid(X, y_train, "random_forest", seed=SEED)
    val_win = hybrid.predict_hybrid(model, hybrid.build_hybrid_features(val_df, lstm_v1_pred["val"]))
    val_pts_pred = hybrid.predict_hybrid(
        model, hybrid.build_hybrid_features(val_df, lstm_v1_pred["val"], points=val_pts))
    return {"train_mae": hybrid.point_metrics(y_train, hybrid.predict_hybrid(model, X))["mae"],
            "window_mae": hybrid.point_metrics(y_val_win, val_win)["mae"],
            "points_mae": hybrid.point_metrics(y_val_pts, val_pts_pred)["mae"],
            "d_importance": float(model.tree.feature_importances_[model.columns.index("d")])}


leak_control = pd.DataFrame({
    "leaky (in-sample d)": control_run(lstm_v1_pred["train"]),
    "out-of-fold d": control_run(lstm42["train"]),
}).T
leak_control["gap_val_minus_train"] = leak_control.window_mae - leak_control.train_mae
leak_control.round(3)